# Solution C1 · Breaking the KNN with noise columns

**Type**: Break the model

## Objective

Add to a two-dimensional problem a growing number of pure noise columns and measure how a KNN collapses while a logistic and a random forest hold up. Measure the **concentration of the distances** (the nearest neighbor
becomes almost as far as the farthest) and repair with feature selection inside the pipeline.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.datasets import make_moons
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline


def generate(d, seed=0):
    X, y = make_moons(3000, noise=0.25, random_state=0)
    noise = np.random.default_rng(seed).normal(size=(len(X), d))
    return train_test_split(np.c_[X, noise], y, test_size=0.5, random_state=0)


def concentration(X, n=300):
    D = np.sqrt(((X[:n, None, :] - X[None, n:2 * n, :]) ** 2).sum(-1))       # distances between two different groups of points
    return D.min(1).mean() / D.max(1).mean()

## Commented solution

In [ ]:
rows = {}
for d in [0, 2, 5, 10, 20, 50, 100, 200]:
    X_tr, X_te, y_tr, y_te = generate(d)
    rows[d] = {"knn": KNeighborsClassifier(5).fit(X_tr, y_tr).score(X_te, y_te),
               "lr": LogisticRegression(max_iter=1000).fit(X_tr, y_tr).score(X_te, y_te),
               "rf": RandomForestClassifier(100, random_state=0, n_jobs=-1).fit(X_tr, y_tr).score(X_te, y_te),
               "concentration": concentration(X_te)}
table = pd.DataFrame(rows).T
table.round(3)

In [ ]:
collapse_d = int(table.index[(table["knn"] < 0.65).to_numpy().argmax()])
loss = {m: table.loc[0, m] - table.loc[200, m] for m in ["knn", "lr", "rf"]}
print("the KNN drops below 0.65 with d =", collapse_d, "| losses:", {m: round(v, 3) for m, v in loss.items()})

In [ ]:
X_tr, X_te, y_tr, y_te = generate(100)
sel = make_pipeline(SelectKBest(f_classif, k=2), KNeighborsClassifier(5)).fit(X_tr, y_tr)
selection_acc = sel.score(X_te, y_te)
chosen_columns = sel[0].get_support().nonzero()[0]
print(f"with d = 100: KNN {table.loc[100, 'knn']:.3f} -> with selection {selection_acc:.3f} | chosen columns: {chosen_columns}")

## Check

In [ ]:
with verify():
    check(list(table.columns) == ["knn", "lr", "rf", "concentration"] and list(table.index) == [0, 2, 5, 10, 20, 50, 100, 200], "table with the four columns for the eight values of d", "index d, columns knn, lr, rf, concentration")
    check(table.loc[0, "knn"] > 0.9 and table.loc[200, "knn"] < 0.62, "the KNN goes from over 0.9 to less than 0.62 (almost chance)", "test accuracy at d = 0 and d = 200")
    check(table.loc[200, "lr"] > 0.8 and table.loc[200, "rf"] > 0.85, "the logistic and the random forest stay above 0.8 and 0.85", "they don't use the distances in the full space")
    check(table["concentration"].is_monotonic_increasing and table.loc[0, "concentration"] < 0.05 and table.loc[200, "concentration"] > 0.7, "the nearest/farthest ratio goes from below 0.05 to above 0.7", "with noise the nearest neighbor is as far as the farthest")
    check(collapse_d in (50, 100), "the KNN drops below 0.65 with 50-100 noise columns", "first d with knn < 0.65")
    check(loss["knn"] > 0.3 and loss["lr"] < 0.06 and loss["rf"] < 0.06, "loss: KNN over 0.3, the others below 0.06", "accuracy(0) - accuracy(200)")
    check(selection_acc > 0.92 and sorted(chosen_columns.tolist()) == [0, 1], "with the selection the accuracy goes back to 0.92 or more and the chosen columns are the first two", "SelectKBest inside the pipeline")
done()

## What you learned

- **Every useless column costs the KNN**: the accuracy drops from 0.93 (without noise) to 0.79 with 10 useless columns, 0.64 with 50 and 0.57 with 200 (chance is 0.5). The logistic does not change (0.83-0.86) and the random forest
  drops a little (0.93 → 0.89): a tree can simply not use the useless columns.
- The explanation is geometric: the squared distance is the sum of the contributions of all the coordinates, and every noise column adds a **random** amount of the same order of magnitude. The ratio between the nearest neighbor and the farthest
  goes from 0.03 to 0.78: with enough dimensions "everyone is equally far" and the neighbor is chosen by the noise.
- The repair is **selecting the features** (here `SelectKBest` with the F test, inside the pipeline to avoid leakage): with 100 noise columns the accuracy goes back to 0.93, equal to the case without noise. In general you need
  feature selection, dimension reduction (PCA, module 6) or a model that selects them by itself.
- Rule of thumb: before a KNN ask yourself **how many of your columns really carry information about the distance** (not about the target). Many columns correlated with the target but little with each other are a good sign; many detail columns are not.

## Common pitfall

Adding all the available features to a KNN "because more information can't hurt". With distance-based models it hurts, silently and in proportion to the number of useless columns.

## Interview question

<details><summary>What is the curse of dimensionality and how does it hit KNN?</summary>

As the number of dimensions grows the space "empties out": the points are almost all at the same distance from each other, the ratio between the nearest neighbor and the farthest tends to 1 and the neighbors stop being meaningful. In KNN it is a direct problem because
the prediction depends only on the distance. It is mitigated with feature selection or reduction, with learned metrics or with models that select them by themselves (trees, regularized linear models), or with a lot more data.

</details>